In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    return response.output_text

In [6]:
question = 'How can I create an account?'
answer = llm(question)
print(answer)

To create an account, usually you would:

1. Go to the service’s website or app.
2. Click **Sign Up**, **Create Account**, or **Register**.
3. Enter the required information, such as:
   - Name
   - Email address or phone number
   - Password
4. Verify your email or phone if prompted.
5. Finish setup by accepting any terms and completing your profile.

If you want, I can give you step-by-step instructions for a specific website or app.


In [11]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)

print(response)

courses_raw = response.json()

documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    print(course)
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

<Response [200]>
{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}
{'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}
{'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}
{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}
{'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 133}
{'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}


1395

In [4]:
import secrets
import string

def generate_short_id(length=8):
    # Combines safe characters: A-Z, a-z, and 0-9
    alphabet = string.ascii_letters + string.digits
    # Securely picks random characters from the alphabet
    return ''.join(secrets.choice(alphabet) for _ in range(length))

# Generate an 8-character ID
short_id = generate_short_id(8)

print(short_id)

9hqH4YRY


In [8]:
import requests

faq_dataset_url = 'https://raw.githubusercontent.com/anilbhaila/llm-zoomcamp-finalproject/refs/heads/main/data/Ecommerce_FAQ_Chatbot_dataset.json'
response = requests.get(faq_dataset_url)

faq_raw = response.json()
faqs = faq_raw.get("questions")

documents = []

for faq in faqs:
    #faq["id"] = generate_short_id(8)
    documents.append(faq)

len(documents)

79

In [9]:
print(documents)

[{'question': 'How can I create an account?', 'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."}, {'question': 'What payment methods do you accept?', 'answer': 'We accept major credit cards, debit cards, and PayPal as payment methods for online orders.'}, {'question': 'How can I track my order?', 'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment."}, {'question': 'What is your return policy?', 'answer': 'Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging. Please refer to our Returns page for detailed instructions.'}, {'question': 'Can I cancel my order?', 'answer': 'You can cancel your order if it has not been shipped yet. Please contact our cus

In [10]:
from minsearch import Index

index = Index(
    text_fields=['question', 'answer']
)

index.fit(documents)

In [11]:
def search(question):
    boost_dict={'question': 2.0}
        
    return index.search(
        question,
        boost_dict=boost_dict,
        num_results=5
    )

In [13]:
question = "How can I create an account?"
search_results = search(question)
search_results

[{'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."},
 {'question': 'Can I order without creating an account?',
  'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.'},
 {'question': 'How can I track my order?',
  'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment."},
 {'question': 'Can I request an invoice for my order?',
  'answer': 'Yes, an invoice is usually included with your order. If you require a separate invoice, please contact our customer support team with your order details.'},
 {'question': 'How can I leave a product review?',
  'answer': "To leave a product review, 

In [14]:
INSTRUCTIONS = '''
Your task is to answer questions from the customer based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."
'''

In [15]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [16]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')
    
    return '\n'.join(lines).strip()

In [17]:
context = build_context(search_results)
print(context)

Q: How can I create an account?
A: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.

Q: Can I order without creating an account?
A: Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.

Q: How can I track my order?
A: You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.

Q: Can I request an invoice for my order?
A: Yes, an invoice is usually included with your order. If you require a separate invoice, please contact our customer support team with your order details.

Q: How can I leave a product review?
A: To leave a product review, navigate to the product page on our website and click on the 'Write a Review' button. You can share your feedback and rating bas

In [18]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [19]:
prompt = build_prompt(question,search_results)
print(prompt)

Question:
How can I create an account?

Context:
Q: How can I create an account?
A: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.

Q: Can I order without creating an account?
A: Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.

Q: How can I track my order?
A: You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.

Q: Can I request an invoice for my order?
A: Yes, an invoice is usually included with your order. If you require a separate invoice, please contact our customer support team with your order details.

Q: How can I leave a product review?
A: To leave a product review, navigate to the product page on our website and click on the 'Write a Review' b

In [20]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [21]:
response.output_text

'To create an account, click on the **“Sign Up”** button in the **top right corner** of the website and follow the instructions to complete the registration process.'

In [22]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_072ae5b95446b534006a6d274aef948199ba8b4b9d1e8d67a0",
  "created_at": 1785538378.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_072ae5b95446b534006a6d274b8f488199b4c847277bb7bb33",
      "content": [
        {
          "annotations": [],
          "text": "To create an account, click on the **“Sign Up”** button in the **top right corner** of the website and follow the instructions to complete the registration process.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1785538379.0,
  "conversation": null,
  "max_output_tokens": null,

In [23]:
input_price = 0.75 / 1000000
output_price = 4.50 / 1000000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00035475000000000003

In [24]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role':'developer', 'content':instructions},
        {'role':'user', 'content':user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [25]:
res = llm(INSTRUCTIONS, prompt)
print(res)

To create an account, click on the **“Sign Up”** button in the top right corner of the website and follow the instructions to complete the registration process.


In [26]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS,prompt,model=model)
    return answer

In [27]:
answer = rag(question)
print(answer)

To create an account, click on the **“Sign Up”** button in the top right corner of the website and follow the instructions to complete the registration process.
